# 02 — Defense demo: the QKV-only case study, end to end

Due to time constraints, this demo reproduces the thesis's **standalone Q–K case study**
(Llama-3-8B, layer 8, GQA group 3) live on Kaggle — the setting where the Flip/ReFlip
mechanism can be observed on exactly the quantity it optimizes.

**Pipeline, one repo script per cell:**
1. `xspot.py` — spot the QKV projections of one GQA group + build the James–Stein
   representative activation from C4 calibration;
2. `fast_quantize_qkv.py` — quantize (RTN) → **Flip** → **Flip + ReFlip** on that group;
3. `tools/summarize_qkv_results.py` + plot scripts — metric table and figures;
4. one final cell that visualizes **all** results together.

Kaggle settings: **GPU T4 x2**, Internet **On**. Total runtime ≈ 25 minutes.

In [ ]:
# --- Cell 1: clone the repository and install dependencies ---
import os, subprocess, sys

if not os.path.exists("/kaggle/tmp/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Itvc0110/Reflip-Flip-on-QKV-.git", "/kaggle/tmp/repo"], check=True)
os.chdir("/kaggle/tmp/repo")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.45,<4.53", "datasets>=2.20,<3.0", "accelerate",
                "sentencepiece", "matplotlib", "seaborn", "tqdm"], check=True)
print("repo + deps ready at", os.getcwd())

In [ ]:
# --- Cell 2: Hugging Face login and download Llama-3-8B (~16 GB, one time) ---
from huggingface_hub import login, snapshot_download

# Paste your token here AT RUN TIME (never save a real token into the notebook).
# Leave "" to use the HF_TOKEN Kaggle Secret instead.
token = ""
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print("(using HF_TOKEN Kaggle Secret)")
login(token=token)

MODEL_DIR = "/kaggle/tmp/models/Llama-3-8B"
snapshot_download(repo_id="meta-llama/Meta-Llama-3-8B", local_dir=MODEL_DIR,
                  ignore_patterns=["original/*", "*.pth"])
print("model at", MODEL_DIR)

## Step 1 — Spot the QKV group and build the representative activation (`xspot.py`)

Loads the full model, hooks the **input of `q_proj` at layer 8** (the hidden state that
actually multiplies $W_Q, W_K, W_V$), runs **128 C4 samples x 512 tokens** of calibration
(65,536 observations over 4,096 channels), applies the **James–Stein-inspired shrinkage**
to get one stable representative vector $x$, and extracts the weights of **GQA group 3**
(4 query heads sharing one K/V head). Takes ~10 minutes.

In [ ]:
# --- Cell 3: run xspot.py ---
import subprocess, sys
subprocess.run([sys.executable, "xspot.py",
                "--model-path", MODEL_DIR,
                "--layer-id", "8", "--group-id", "3",
                "--n-samples", "128", "--seqlen", "512",
                "--output-dir", "/kaggle/working/xspot_layer8_group3"], check=True)

import os
print("\nexported files:")
for f in sorted(os.listdir("/kaggle/working/xspot_layer8_group3")):
    print("  ", f)

## Step 2 — Quantize → Flip → Flip + ReFlip (`fast_quantize_qkv.py`)

On the exported group: group-wise asymmetric **INT4** (RTN start), then **Flip**
(one-level integer moves accepted only when the activation-weighted row residual strictly
shrinks, with a Kneedle mask protecting extreme activation channels), then **ReFlip**
(query-only moves, shared quantized key held fixed, accepted only when the scalar Q–K
score moves strictly closer to full precision; Kneedle-gated moderate region). Prints the
per-head error table and saves `quantization_results.npz` + two analysis figures.
Runs in ~1–2 minutes (NumPy).

In [ ]:
# --- Cell 4: run the quantization + refinement pipeline ---
import subprocess, sys
subprocess.run([sys.executable, "fast_quantize_qkv.py",
                "--data-dir", "/kaggle/working/xspot_layer8_group3",
                "--group-id", "3",
                "--critical-dim-pct", "0.1",
                "--output-dir", "/kaggle/working/qkv_results"], check=True)

In [ ]:
# --- Cell 5: clean metric table (matches thesis Tables 4.2/4.3) ---
import subprocess, sys
subprocess.run([sys.executable, "tools/summarize_qkv_results.py",
                "--npz", "/kaggle/working/qkv_results/quantization_results.npz"], check=True)

In [ ]:
# --- Cell 6: regenerate the two Kneedle figures from this run (real data) ---
import subprocess, sys
NPZ = "/kaggle/working/qkv_results/quantization_results.npz"
subprocess.run([sys.executable, "tools/plot_flip_activation_kneedle.py",
                "--npz", NPZ, "--out", "/kaggle/working/qkv_results/flip_kneedle.png"], check=True)
subprocess.run([sys.executable, "tools/plot_kneedle_sensitivity.py",
                "--npz", NPZ, "--out", "/kaggle/working/qkv_results/reflip_kneedle.png"], check=True)

## Step 3 — All results in one view

Left to right, the story: RTN leaves a large scalar Q–K error → Flip halves it →
ReFlip cuts it further, on every query head — with the Kneedle figures showing *where*
each stage was allowed to act.

In [ ]:
# --- Cell 7: visualize everything ---
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

d = np.load("/kaggle/working/qkv_results/quantization_results.npz", allow_pickle=True)
err_n, err_f, err_r = d["errors_nearest"], d["errors_flip"], d["errors_reflip"]
heads = np.arange(len(err_n))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
w = 0.27
axes[0].bar(heads - w, np.abs(err_n), w, label="RTN", color="#1f77b4")
axes[0].bar(heads,     np.abs(err_f), w, label="Flip", color="#ff7f0e")
axes[0].bar(heads + w, np.abs(err_r), w, label="Flip + ReFlip", color="#2ca02c")
axes[0].set_xticks(heads); axes[0].set_xticklabels([f"head {h}" for h in heads])
axes[0].set_ylabel("|scalar Q-K surrogate error|")
axes[0].set_title("Per-head error: RTN -> Flip -> Flip + ReFlip")
axes[0].legend(); axes[0].grid(True, axis="y", alpha=0.3)

mae = [np.mean(np.abs(err_n)), np.mean(np.abs(err_f)), np.mean(np.abs(err_r))]
labels = ["RTN", "Flip", "Flip + ReFlip"]
bars = axes[1].bar(labels, mae, color=["#1f77b4", "#ff7f0e", "#2ca02c"])
for b, v in zip(bars, mae):
    axes[1].annotate(f"{v:.4f}", (b.get_x() + b.get_width() / 2, v),
                     ha="center", va="bottom", fontsize=10)
axes[1].annotate(f"-{(1 - mae[1] / mae[0]) * 100:.1f}%", (1, mae[1] / 2), ha="center",
                 color="white", fontsize=11, fontweight="bold")
axes[1].annotate(f"-{(1 - mae[2] / mae[0]) * 100:.1f}%", (2, mae[2] / 2), ha="center",
                 color="white", fontsize=11, fontweight="bold")
axes[1].set_ylabel("mean |error| (MAE)")
axes[1].set_title("Aggregate error vs RTN")
axes[1].grid(True, axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

for png in ["flip_kneedle.png", "reflip_kneedle.png",
            "attention_quantization_analysis.png", "sorted_error_comparison.png"]:
    display(Image(f"/kaggle/working/qkv_results/{png}", width=880))

## Wrap-up

- **RTN** is optimal per weight, yet leaves a large scalar Q–K error — because the error
  that matters is activation-weighted, signed, and coupled through $QK^\top$.
- **Flip** roughly halves that error by revisiting ~0.8% of the rounding decisions
  (each accepted only when the row residual strictly shrinks).
- **ReFlip** adds a few dozen query-only moves (shared key fixed) and cuts the remaining
  error by another ~30%, on every head.
- All numbers on screen are regenerated live and match thesis Tables 4.2/4.3
  (same pipeline, same seed); the full-model evaluation (Tables 4.4–4.7) is reported in
  the thesis.